<a href="https://colab.research.google.com/github/isil-ada/GML_H1/blob/main/GML_H1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **PHASE 1: Data Acquisition & Preprocessing**

In [1]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings("ignore")

**── 1.1 Veri Yükleme ─────────────────────────────────────────**


In [2]:
emails_raw = pd.read_csv("/content/emails.csv", engine="python", on_bad_lines="skip")
print(f"Ham veri: {emails_raw.shape}")

emails_raw["folder"] = emails_raw["file"].apply(
    lambda x: x.split("/")[1].lower() if isinstance(x, str) and len(x.split("/")) > 1 else "unknown"
)
print(emails_raw["folder"].value_counts().head(20))

Ham veri: (843222, 2)
folder
unknown               216820
all_documents         127986
discussion_threads     58574
sent                   57643
deleted_items          51331
inbox                  44838
sent_items             37915
notes_inbox            36595
_sent_mail             30088
                       14970
calendar                6113
archiving               4457
ou=3dhou                4177
_americas               4005
ou=hou                  3179
personal                2577
attachments             2025
meetings                1872
omnicalendarentry>      1776
omniorgtable>           1775
Name: count, dtype: int64


**── 1.2 Cleaning ─────────────────────────────────────────────**

In [3]:
def clean_email(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(
        r"^(Message-ID|Date|From|To|Subject|Cc|Bcc|Mime-Version|"
        r"Content-Type|Content-Transfer-Encoding|X-[\w-]+|Return-Path|"
        r"Received|In-Reply-To|References|Thread-Index|Thread-Topic)"
        r":.*$",
        "", text, flags=re.MULTILINE | re.IGNORECASE
    )
    text = re.sub(r"-{3,}.*?-{3,}", "", text, flags=re.DOTALL)
    text = re.sub(r"_{3,}.*", "", text, flags=re.DOTALL)
    text = re.sub(
        r"(best regards?|thanks?|sincerely|cheers|regards?|"
        r"yours? truly|warm regards?)[^\n]*(\n.*){0,4}$",
        "", text, flags=re.IGNORECASE
    )
    text = re.sub(r"\s+", " ", text).strip()
    return text

**── 1.3 Labeling ─────────────────────────────────────────────**

In [4]:
professional = {"sent", "sent_items", "_sent_mail", "all_documents", "discussion_threads"}
personal     = {"personal", "inbox", "notes_inbox"}

emails_raw = emails_raw[emails_raw["folder"].isin(professional | personal)].copy()
emails_raw["label"] = emails_raw["folder"].apply(lambda x: 1 if x in personal else 0)

emails_raw["body"] = emails_raw["message"].apply(clean_email)
emails_raw = emails_raw[emails_raw["body"].str.len() > 20].reset_index(drop=True)

print(f"Temizlenmiş veri: {emails_raw.shape} \n")

Temizlenmiş veri: (387845, 5) 



In [5]:
emails_raw["label"].value_counts()

,count
label,
0,304598
1,83247


In [6]:
emails_raw[["file", "body", "label", "folder"]].head(3)

,file,body,label,folder
0,allen-p/_sent_mail/10.,Traveling to have a business meeting takes the...,0,_sent_mail
1,allen-p/_sent_mail/100.,test successful. way to go!!!,0,_sent_mail
2,allen-p/_sent_mail/1000.,"Randy, Can you send me a schedule of the salar...",0,_sent_mail


**── 1.4 Sample ───────────────────────────────────────────────**

In [7]:
sample = emails_raw.groupby("label", group_keys=False).apply(
    lambda x: x.sample(min(len(x), 2500), random_state=42)
).reset_index(drop=True)

print(f"Sample: {len(sample)}")
print(sample["label"].value_counts())

Sample: 5000
label
0    2500
1    2500
Name: count, dtype: int64


**── 1.5 Update Sample (dengeli, DeprecationWarning yok) ─────────────**

# **PHASE 2: Graph Construction (NER & BoW)**



In [8]:
import spacy
import networkx as nx
from collections import defaultdict, Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

**── 2.1 Graf Başlat ──────────────────────────────────────────**


In [9]:
G = nx.Graph()
G.add_nodes_from(sample.index)

**── 2.2 NER Edge'leri (spaCy) ────────────────────────────────**

In [10]:
print("NER edge'leri hesaplanıyor...")
nlp_ner = spacy.load("en_core_web_sm", disable=["tok2vec","tagger","parser","lemmatizer","attribute_ruler"])
texts = [str(t)[:1500] for t in sample["body"]]

entities = [set() for _ in range(len(sample))]
for i, doc in enumerate(nlp_ner.pipe(texts, batch_size=64)):
    entities[i] = set(ent.text.lower() for ent in doc.ents if ent.label_ in {"PERSON","ORG","GPE"})
    if i % 500 == 0:
        print(f"  NER: {i}/{len(sample)}")

entity_to_emails = defaultdict(list)
for idx, ent_set in enumerate(entities):
    for ent in ent_set:
        entity_to_emails[ent].append(idx)

pair_count = Counter()
for ent, idxs in entity_to_emails.items():
    if len(idxs) < 2 or len(idxs) > 500:
        continue
    for i in range(len(idxs)):
        for j in range(i+1, len(idxs)):
            pair_count[(idxs[i], idxs[j])] += 1

ner_edges = 0
for (i, j), count in pair_count.items():
    if count >= 2:
        G.add_edge(i, j, type="ner")
        ner_edges += 1

print(f"NER edge sayısı: {ner_edges}")

NER edge'leri hesaplanıyor...
  NER: 0/5000
  NER: 500/5000
  NER: 1000/5000
  NER: 1500/5000
  NER: 2000/5000
  NER: 2500/5000
  NER: 3000/5000
  NER: 3500/5000
  NER: 4000/5000
  NER: 4500/5000
NER edge sayısı: 25156


**── 2.3 BoW Edge'leri (Cosine Similarity > 0.7) ──────────────**

In [11]:
print("BoW edge'leri hesaplanıyor...")

vectorizer = TfidfVectorizer(max_features=5000, stop_words="english")
bow_matrix = vectorizer.fit_transform(sample["body"])

BATCH = 200
bow_edges = 0
for start in range(0, len(sample), BATCH):
    end = min(start + BATCH, len(sample))
    batch_sim = cosine_similarity(bow_matrix[start:end], bow_matrix)
    for local_i, global_i in enumerate(range(start, end)):
        for global_j in range(global_i + 1, len(sample)):
            if batch_sim[local_i, global_j] > 0.7:
                if not G.has_edge(global_i, global_j):
                    G.add_edge(global_i, global_j, type="bow")
                bow_edges += 1
    if start % 2000 == 0:
        print(f"  BoW: {start}/{len(sample)}")

print(f"BoW edge sayısı: {bow_edges}")

BoW edge'leri hesaplanıyor...
  BoW: 0/5000
  BoW: 2000/5000
  BoW: 4000/5000
BoW edge sayısı: 36422


**── 2.4 Graf Özeti ───────────────────────────────────────────**

In [12]:
print(f"\n── Graf Özeti ──")
print(f"Node sayısı : {G.number_of_nodes()}")
print(f"Edge sayısı : {G.number_of_edges()}")
print(f"  NER edges : {ner_edges}")
print(f"  BoW edges : {bow_edges}")
print(f"Bağlantılı mı: {nx.is_connected(G)}")
print(f"Ortalama derece: {sum(dict(G.degree()).values()) / G.number_of_nodes():.2f}")


── Graf Özeti ──
Node sayısı : 5000
Edge sayısı : 56037
  NER edges : 25156
  BoW edges : 36422
Bağlantılı mı: False
Ortalama derece: 22.41


# **PHASE 3: Embedding Techniques**


In [13]:
!pip install spacy scikit-learn networkx gensim node2vec torch torch-geometric -q
!python -m spacy download en_core_web_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 34.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [14]:
import random
from gensim.models import Word2Vec
from node2vec import Node2Vec
from sklearn.decomposition import PCA
import scipy.sparse as sp
import torch
import torch.nn as nn
import torch.nn.functional as Ff
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv

**── 3.1 DeepWalk ─────────────────────────────────────────────**


In [15]:
def deep_walk(graph, num_walks=10, walk_length=10):
    walks = []
    nodes = list(graph.nodes())
    for _ in range(num_walks):
        random.shuffle(nodes)
        for node in nodes:
            walk = [str(node)]
            current = node
            for _ in range(walk_length - 1):
                neighbors = list(graph.neighbors(current))
                if neighbors:
                    current = random.choice(neighbors)
                    walk.append(str(current))
                else:
                    break
            walks.append(walk)
    return walks

print("DeepWalk: Random walk'lar oluşturuluyor...")
dw_walks = deep_walk(G, num_walks=10, walk_length=10)

print("DeepWalk: Skip-gram modeli eğitiliyor...")
dw_model = Word2Vec(sentences=dw_walks, vector_size=64, window=5,
                    min_count=0, sg=1, workers=4, epochs=5)

dw_embeddings = np.array([dw_model.wv[str(i)] for i in range(len(sample))])
print(f"DeepWalk embeddings shape: {dw_embeddings.shape}")

DeepWalk: Random walk'lar oluşturuluyor...
DeepWalk: Skip-gram modeli eğitiliyor...
DeepWalk embeddings shape: (5000, 64)


**── 3.2 Node2Vec ─────────────────────────────────────────────**


In [16]:
print("Node2Vec: Eğitiliyor...")
node2vec = Node2Vec(G, dimensions=64,
                    walk_length=10,
                    num_walks=10,
                    p=1,
                    q=0.5,
                    workers=1,
                    quiet=True)

n2v_model = node2vec.fit(window=5, min_count=0, sg=1, epochs=5)

n2v_embeddings = np.array([
    n2v_model.wv[str(i)] for i in range(len(sample))
])

print(f"Node2Vec embeddings shape: {n2v_embeddings.shape}")

Node2Vec: Eğitiliyor...
Node2Vec embeddings shape: (5000, 64)


**── 3.3 SiGraC (Signed Graph Embedding) ─────────────────────**

In [17]:
print("SiGraC: Signed embedding hazırlanıyor...")

pos_edges, neg_edges = [], []
for u, v, d in G.edges(data=True):
    (pos_edges if d.get("type") == "ner" else neg_edges).append((u, v))

print(f"  Pozitif edge: {len(pos_edges)}")
print(f"  Negatif edge: {len(neg_edges)}")

n = len(sample)

def build_sparse_adj(edges, n):
    if not edges:
        return sp.csr_matrix((n, n), dtype=np.float32)
    rows, cols = zip(*edges)
    rows, cols = list(rows) + list(cols), list(cols) + list(rows)
    data = np.ones(len(rows), dtype=np.float32)
    return sp.csr_matrix((data, (rows, cols)), shape=(n, n))

def normalize_sparse_adj(A):
    degree = np.array(A.sum(axis=1)).flatten() + 1e-9
    D_inv_sqrt = sp.diags(1.0 / np.sqrt(degree))
    return D_inv_sqrt @ A @ D_inv_sqrt

A_pos_norm = normalize_sparse_adj(build_sparse_adj(pos_edges, n))
A_neg_norm = normalize_sparse_adj(build_sparse_adj(neg_edges, n))

# Sparse matrix multiplication
H_signed = A_pos_norm @ bow_matrix - 0.5 * (A_neg_norm @ bow_matrix)
H_signed = H_signed.toarray().astype(np.float32)

pca = PCA(n_components=64, random_state=42)
sigrac_embeddings = pca.fit_transform(H_signed)
print(f"SiGraC embeddings shape: {sigrac_embeddings.shape}")

SiGraC: Signed embedding hazırlanıyor...
  Pozitif edge: 25156
  Negatif edge: 30881
SiGraC embeddings shape: (5000, 64)


**── 3.4 GNN (2-layer GCN) ────────────────────────────────────**

In [18]:
print("GNN: PyTorch Geometric GCN eğitiliyor...")

edge_index = torch.tensor(list(G.edges()), dtype=torch.long).t().contiguous()
edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)

x = torch.tensor(bow_matrix.toarray(), dtype=torch.float)
y = torch.tensor(sample["label"].values, dtype=torch.long)

n = len(sample)
perm = torch.randperm(n)
train_mask = torch.zeros(n, dtype=torch.bool)
test_mask  = torch.zeros(n, dtype=torch.bool)
train_mask[perm[:int(0.8 * n)]] = True
test_mask[perm[int(0.8 * n):]]  = True

data = Data(x=x, edge_index=edge_index, y=y,
            train_mask=train_mask, test_mask=test_mask)

class GCN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = GCN(in_channels=x.shape[1], hidden_channels=64, out_channels=2).to(device)
data   = data.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

print("GCN eğitimi başlıyor...")
model.train()
for epoch in range(1, 101):
    optimizer.zero_grad()
    out  = model(data.x, data.edge_index)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    if epoch % 20 == 0:
        print(f"  Epoch {epoch:3d} | Loss: {loss.item():.4f}")

model.eval()
with torch.no_grad():
    gcn_embeddings = model.conv1(data.x, data.edge_index)
    gcn_embeddings = F.relu(gcn_embeddings).cpu().numpy()

print(f"GCN embeddings shape: {gcn_embeddings.shape}")
print("✅ Phase 3 tamamlandı!")

GNN: PyTorch Geometric GCN eğitiliyor...
GCN eğitimi başlıyor...
  Epoch  20 | Loss: 0.4081
  Epoch  40 | Loss: 0.3563
  Epoch  60 | Loss: 0.3285
  Epoch  80 | Loss: 0.3134
  Epoch 100 | Loss: 0.3069
GCN embeddings shape: (5000, 64)
✅ Phase 3 tamamlandı!


# **PHASE 4: Classification & Evaluation**

In [19]:
import time
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

**4.1 Değerlendirme Fonksiyonları**

In [20]:
y = sample["label"].values

def evaluate_embedding(name, embeddings, y):
    X_train, X_test, y_train, y_test = train_test_split(
        embeddings, y, test_size=0.2, random_state=42, stratify=y
    )
    clf = LogisticRegression(max_iter=1000, random_state=42)
    start = time.time()
    clf.fit(X_train, y_train)
    train_time = time.time() - start
    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, average="weighted")
    print(f"\n── {name} ──")
    print(classification_report(y_test, y_pred))
    return {"Method": name, "Accuracy": round(acc, 4),
            "F1-Score": round(f1, 4), "Time to Train (s)": round(train_time, 2)}

def evaluate_gcn(model, data):
    model.eval()
    start = time.time()
    with torch.no_grad():
        out  = model(data.x, data.edge_index)
        pred = out.argmax(dim=1)
    train_time = time.time() - start
    y_true = data.y[data.test_mask].cpu().numpy()
    y_pred = pred[data.test_mask].cpu().numpy()
    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, average="weighted")
    print(f"\n── GNN (GCN) ──")
    print(classification_report(y_true, y_pred))
    return {"Method": "GNN (GCN)", "Accuracy": round(acc, 4),
            "F1-Score": round(f1, 4), "Time to Train (s)": round(train_time, 2)}

**── 4.2 Karşılaştırma Tablosu ────────────────────────────────**


In [21]:
results = []
results.append(evaluate_embedding("DeepWalk",  dw_embeddings,     y))
results.append(evaluate_embedding("Node2Vec",  n2v_embeddings,    y))
results.append(evaluate_embedding("SiGraC",    sigrac_embeddings, y))
results.append(evaluate_gcn(model, data))

df_results = pd.DataFrame(results)
print("\n" + "="*55)
print("       EMBEDDING YÖNTEMLERİ KARŞILAŞTIRMA TABLOSU")
print("="*55)
print(df_results.to_string(index=False))
print("="*55)


── DeepWalk ──
              precision    recall  f1-score   support

           0       0.60      0.80      0.69       500
           1       0.70      0.46      0.55       500

    accuracy                           0.63      1000
   macro avg       0.65      0.63      0.62      1000
weighted avg       0.65      0.63      0.62      1000


── Node2Vec ──
              precision    recall  f1-score   support

           0       0.59      0.81      0.69       500
           1       0.70      0.44      0.54       500

    accuracy                           0.63      1000
   macro avg       0.65      0.63      0.61      1000
weighted avg       0.65      0.63      0.61      1000


── SiGraC ──
              precision    recall  f1-score   support

           0       0.58      0.83      0.68       500
           1       0.70      0.40      0.51       500

    accuracy                           0.61      1000
   macro avg       0.64      0.61      0.60      1000
weighted avg       0.64     